# VGGT-1B on Kaggle: re-implementation check and claim verification

Settings: **Accelerator: GPU T4 x2** (or P100), **Internet: on**, **Persistence: Files only**.
Run everything with *Save Version > Save & Run All (Commit)*; outputs are zipped to `/kaggle/working/vggt_results.zip`.

The repository re-implements the official `facebookresearch/vggt` package (same layout and API) and uses Meta's
released `facebook/VGGT-1B` weights unchanged. VGGT-1B is not retrained. This notebook checks:

1. **Implementation**: identical outputs to the official code (random weights and released weights, all heads incl. tracking).
2. **Speed**: inference time and memory for 1 to 32 frames (paper: feed-forward, about a second for tens of frames on an H100).
3. **It works on real images**: cameras, depth, point maps and tracks on the official example scenes, with an interactive 3D viewer.
4. **Accuracy against exact ground truth** on rendered scenes: camera pose (AUC), depth, point maps (point head vs. depth + camera),
   compared with a classical SIFT + 5-point RANSAC baseline.

In [ ]:
BRANCH = "claude/clever-curie-rl6bco"   # branch with the mirrored package; use "main" once it is merged
!pip -q install huggingface_hub einops safetensors
# Fresh clones on every run ("Persistence: Files only" keeps old copies; the 5 GB weights in hf_cache are kept)
!rm -rf /kaggle/working/vggt_official /kaggle/working/vggt_ours /kaggle/working/results
!git clone -q https://github.com/facebookresearch/vggt /kaggle/working/vggt_official
!git clone -q -b {BRANCH} https://github.com/Platinum-Saber/VGGT /kaggle/working/vggt_ours
OFFICIAL, OURS, OUT = "/kaggle/working/vggt_official", "/kaggle/working/vggt_ours", "/kaggle/working/results"

import os, glob, json, sys, time, importlib, torch
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs(OUT, exist_ok=True)
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download("facebook/VGGT-1B", "model.pt")
device = "cuda"
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(torch.__version__, torch.cuda.get_device_name(), dtype)

## 1. Implementation check against the official code

In [ ]:
# Random weights, reduced config with every component (DINOv2 patchifier, RoPE, AA, camera, DPT, track heads),
# plus identical parameter names/shapes for the full VGGT-1B config.
!cd {OURS} && VGGT_OFFICIAL={OFFICIAL} python -m pytest tests -q -p no:warnings

In [ ]:
def use_pkg(root):
    # Both packages are called `vggt`: drop the loaded one and put `root` first on sys.path.
    for k in [k for k in sys.modules if k == "vggt" or k.startswith("vggt.")]:
        del sys.modules[k]
    sys.path[:] = [root] + [p for p in sys.path if p not in (OFFICIAL, OURS)]
    importlib.invalidate_caches()

paths = sorted(glob.glob(f"{OFFICIAL}/examples/kitchen/images/*"))[:8]
query = torch.tensor([[100., 150.], [260., 200.], [400., 120.], [450., 300.]], device=device)
keys = ["pose_enc", "depth", "depth_conf", "world_points", "world_points_conf", "track", "vis", "conf"]
outs = {}
for name, root in [("official", OFFICIAL), ("ours", OURS)]:
    use_pkg(root)
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images
    images = load_and_preprocess_images(paths).to(device)
    model = VGGT(); model.load_state_dict(torch.load(ckpt, map_location="cpu"), strict=True)
    model = model.to(device).eval()
    with torch.no_grad():                      # fp32, released weights
        outs[name] = {k: v.float().cpu() for k, v in model(images, query_points=query).items() if k in keys}
    del model; torch.cuda.empty_cache()
eq = {}
for k in keys:
    d = (outs["ours"][k] - outs["official"][k]).abs()
    eq[k] = {"max_abs_diff": d.max().item(), "max_abs_ref": outs["official"][k].abs().max().item()}
    print(f"{k:18s} max|diff|={eq[k]['max_abs_diff']:.2e}  max|ref|={eq[k]['max_abs_ref']:.2e}")
json.dump({"images": len(paths), "image_hw": list(images.shape[-2:]), "gpu": torch.cuda.get_device_name(), "outputs": eq},
          open(f"{OUT}/equivalence_vggt1b.json", "w"), indent=1)

## 2. Inference time and memory

In [ ]:
use_pkg(OURS)
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
model = VGGT(); model.load_state_dict(torch.load(ckpt, map_location="cpu")); model = model.to(device).eval()
frames_all = sorted(glob.glob(f"{OFFICIAL}/examples/room/images/*")) + sorted(glob.glob(f"{OFFICIAL}/examples/kitchen/images/*"))
timing = []
for n in [1, 2, 4, 8, 16, 32]:
    if n > len(frames_all): break
    x = load_and_preprocess_images(frames_all[:n]).to(device)
    for rep in range(2):                       # first run includes CUDA warm-up; keep the second
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t = time.time()
        with torch.no_grad(), torch.autocast("cuda", dtype=dtype):
            model(x)
        torch.cuda.synchronize(); sec = time.time() - t
    mem = torch.cuda.max_memory_allocated() / 2**30
    timing.append({"frames": n, "seconds": round(sec, 3), "peak_mem_gib": round(mem, 2)})
    print(f"{n:3d} frames: {sec:6.2f} s   peak memory {mem:5.1f} GiB")
json.dump({"gpu": torch.cuda.get_device_name(), "dtype": str(dtype), "image_hw": list(x.shape[-2:]), "timing": timing},
          open(f"{OUT}/timing.json", "w"), indent=1)
del model; torch.cuda.empty_cache()

## 3. Real images: cameras, depth, points, tracks, and the 3D viewer

`demo_pretrained.py` writes depth maps, `points.ply`, `cameras.json`, track visualisations and a self-contained
`viewer.html` per scene. The viewer is shown inline below; it is also inside the output zip (open it in a browser).

In [ ]:
scenes = {"kitchen": 25, "room": 0, "llff_fern": 0, "llff_flower": 0}   # 0 = all images
for scene, n in scenes.items():
    !cd {OURS} && python demo_pretrained.py --checkpoint {ckpt} --images {OFFICIAL}/examples/{scene}/images --max-frames {n} --out {OUT}/demo_{scene}

In [ ]:
import html
from IPython.display import HTML, Image, display
def show_viewer(scene, height=620):
    src = open(f"{OUT}/demo_{scene}/viewer.html", encoding="utf-8").read()
    display(HTML(f'<iframe srcdoc="{html.escape(src)}" style="width:100%;height:{height}px;border:0"></iframe>'))
show_viewer("kitchen")
display(Image(f"{OUT}/demo_kitchen/tracks/tracks_grid.png", width=900))

In [ ]:
show_viewer("room")

## 4. Accuracy against exact ground truth (rendered scenes)

Real benchmarks (CO3D, RealEstate10K) are too large for a Kaggle session, so we render test scenes with exact depth,
points and cameras (`evaluation/synthetic.py`, 518 x 518, 8 views each) and evaluate VGGT-1B **zero-shot** with the
paper's metrics: pairwise pose AUC@30 / RRA / RTA, depth AbsRel and delta<1.25, point error after Sim(3) alignment
(point head vs. depth + predicted cameras), and the classical SIFT + 5-point RANSAC baseline (given the GT intrinsics).

In [ ]:
!cd {OURS} && python -m evaluation.evaluate_pretrained --checkpoint {ckpt} --num-scenes 50 --frames 2 4 8 --sift --data /kaggle/working/data/test_518.pt --out {OUT}/eval_pretrained.json
print(json.dumps(json.load(open(f"{OUT}/eval_pretrained.json")), indent=1))

In [ ]:
# Zip the small outputs (no weights, no rendered data) for download.
!cd /kaggle/working && rm -f vggt_results.zip && zip -qr vggt_results.zip results -x "*.npz" && ls -la vggt_results.zip